# 07 — Final Holdout + Probability Calibration

## Objective
Freeze the selected CatBoost candidate from notebook 06, calibrate its raw probabilities using **validation only**, then evaluate **test exactly once**.

Selected candidate: `catboost_interactions_unweighted` from run `20261004_222052`.

### Rules
- No retraining or model selection on test.
- Validation is used to fit the probability calibrator.
- Test is opened once for final performance reporting.
- A guard artifact prevents accidental re-evaluation on reruns.


## 1. Install dependencies


In [1]:
!pip -q install catboost pyarrow scikit-learn joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.3 MB/s eta 0:00:00


## 2. Mount Drive and imports


In [2]:
from pathlib import Path
import sys, json, shutil
import numpy as np
import pandas as pd
import joblib
from google.colab import drive
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss

drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/03 Projects/Factored Hackathon')
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from propensity.config import TARGET, DATASET_PATH, ARTIFACT_DIR, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from propensity.data import load_dataset, temporal_split, make_xy
from propensity.features import add_propensity_interactions, INTERACTION_FEATURES
from propensity.catboost_modeling import prepare_catboost_frame
from propensity.evaluation import propensity_deciles

CANDIDATE_RUN_ID = '20261004_222052'
CANDIDATE_NAME = 'catboost_interactions_unweighted'
CANDIDATE_DIR = Path(ARTIFACT_DIR) / 'catboost_experiments' / CANDIDATE_RUN_ID / CANDIDATE_NAME
MODEL_PATH = CANDIDATE_DIR / 'model.joblib'

FINAL_DIR = Path(ARTIFACT_DIR) / 'final_evaluation' / f'catboost_{CANDIDATE_RUN_ID}'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
GUARD_PATH = FINAL_DIR / 'FINAL_TEST_EVALUATED.json'

print('Candidate:', MODEL_PATH)
print('Final artifacts:', FINAL_DIR)
assert MODEL_PATH.exists(), f'Missing selected model: {MODEL_PATH}'


Mounted at /content/drive
Candidate: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/catboost_experiments/20261004_222052/catboost_interactions_unweighted/model.joblib
Final artifacts: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/final_evaluation/catboost_20261004_222052


## 3. Load the same dataset and temporal split


In [3]:
df = load_dataset(DATASET_PATH)
df = add_propensity_interactions(df)

CATEGORICAL_FEATURES_CB = CATEGORICAL_FEATURES + INTERACTION_FEATURES
MODEL_FEATURES_CB = NUMERIC_FEATURES + CATEGORICAL_FEATURES_CB

split = temporal_split(df)
X_val, y_val = make_xy(split.validation_df, MODEL_FEATURES_CB, TARGET)
X_test, y_test = make_xy(split.test_df, MODEL_FEATURES_CB, TARGET)

X_val_cb = prepare_catboost_frame(X_val, CATEGORICAL_FEATURES_CB)
X_test_cb = prepare_catboost_frame(X_test, CATEGORICAL_FEATURES_CB)

print('Validation rows:', f'{len(X_val):,}', 'base rate:', f'{y_val.mean():.4%}')
print('Test rows:', f'{len(X_test):,}', 'base rate:', f'{y_test.mean():.4%}')


Validation rows: 182,104 base rate: 0.7457%
Test rows: 182,105 base rate: 0.6908%


## 4. Load the frozen model and fit calibration on validation only


In [4]:
model = joblib.load(MODEL_PATH)

val_raw = model.predict_proba(X_val_cb)[:, 1]
EPS = 1e-8
val_logit = np.log(np.clip(val_raw, EPS, 1-EPS) / (1 - np.clip(val_raw, EPS, 1-EPS))).reshape(-1, 1)

platt = LogisticRegression(solver='lbfgs', max_iter=1000)
platt.fit(val_logit, y_val)
val_cal = platt.predict_proba(val_logit)[:, 1]

val_summary = {
    'raw_log_loss': float(log_loss(y_val, val_raw, labels=[0,1])),
    'calibrated_log_loss': float(log_loss(y_val, val_cal, labels=[0,1])),
    'raw_brier': float(brier_score_loss(y_val, val_raw)),
    'calibrated_brier': float(brier_score_loss(y_val, val_cal)),
    'roc_auc': float(roc_auc_score(y_val, val_raw)),
    'pr_auc': float(average_precision_score(y_val, val_raw)),
    'base_rate': float(y_val.mean()),
}
print(pd.Series(val_summary))
joblib.dump(platt, FINAL_DIR / 'platt_calibrator.joblib')


raw_log_loss           0.043845
calibrated_log_loss    0.043843
raw_brier              0.007400
calibrated_brier       0.007400
roc_auc                0.552135
pr_auc                 0.008918
base_rate              0.007457
dtype: float64


['/content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/final_evaluation/catboost_20261004_222052/platt_calibrator.joblib']

## 5. FINAL TEST — run exactly once


In [5]:
if GUARD_PATH.exists():
    print('FINAL TEST was already evaluated for this candidate. Reusing saved results; test will NOT be scored again.')
    saved = json.loads(GUARD_PATH.read_text())
    display(pd.Series(saved['test_metrics']))
    test_deciles = pd.read_csv(FINAL_DIR / 'test_deciles.csv', index_col=0)
    display(test_deciles)
else:
    test_raw = model.predict_proba(X_test_cb)[:, 1]
    test_logit = np.log(np.clip(test_raw, EPS, 1-EPS) / (1 - np.clip(test_raw, EPS, 1-EPS))).reshape(-1, 1)
    test_cal = platt.predict_proba(test_logit)[:, 1]

    test_metrics = {
        'raw_roc_auc': float(roc_auc_score(y_test, test_raw)),
        'raw_pr_auc': float(average_precision_score(y_test, test_raw)),
        'base_rate': float(y_test.mean()),
        'pr_auc_lift_vs_base': float(average_precision_score(y_test, test_raw) / y_test.mean()),
        'raw_log_loss': float(log_loss(y_test, test_raw, labels=[0,1])),
        'calibrated_log_loss': float(log_loss(y_test, test_cal, labels=[0,1])),
        'raw_brier': float(brier_score_loss(y_test, test_raw)),
        'calibrated_brier': float(brier_score_loss(y_test, test_cal)),
    }

    scored = split.test_df.copy()
    scored['propensity_raw'] = test_raw
    scored['propensity_calibrated'] = test_cal
    scored['prediction'] = test_raw
    test_deciles = propensity_deciles(scored[TARGET], scored['prediction'])

    top = test_deciles.loc[test_deciles.index.max()]
    test_metrics['top_decile_conversion_rate'] = float(top['observed_conversion_rate'])
    test_metrics['top_decile_lift_vs_random'] = float(top['lift_vs_random'])

    scored.to_parquet(FINAL_DIR / 'scored_test.parquet', index=False)
    test_deciles.to_csv(FINAL_DIR / 'test_deciles.csv')
    joblib.dump(model, FINAL_DIR / 'model.joblib')

    payload = {
        'candidate_run_id': CANDIDATE_RUN_ID,
        'candidate_name': CANDIDATE_NAME,
        'validation_calibration': val_summary,
        'test_metrics': test_metrics,
        'model_features': MODEL_FEATURES_CB,
        'categorical_features': CATEGORICAL_FEATURES_CB,
        'note': 'Final holdout evaluated once. Do not use test for further model selection.'
    }
    GUARD_PATH.write_text(json.dumps(payload, indent=2), encoding='utf-8')

    print('FINAL TEST METRICS')
    display(pd.Series(test_metrics))
    print('TEST DECILES')
    display(test_deciles)
    print('Saved:', FINAL_DIR)


FINAL TEST METRICS


,0
raw_roc_auc,0.574473
raw_pr_auc,0.008602
base_rate,0.006908
pr_auc_lift_vs_base,1.245216
raw_log_loss,0.041012
calibrated_log_loss,0.041018
raw_brier,0.006857
calibrated_brier,0.006857
top_decile_conversion_rate,0.009225
top_decile_lift_vs_random,1.335416


TEST DECILES


,rows,conversions,observed_conversion_rate,avg_propensity,lift_vs_random
decile,,,,,
10,18211,168,0.009225,0.009405,1.335416
9,18210,179,0.009830,0.008857,1.422933
8,18211,151,0.008292,0.008393,1.200285
7,18210,156,0.008567,0.007778,1.240098
6,18210,136,0.007468,0.006813,1.081111
5,18211,96,0.005272,0.006120,0.763095
4,18210,91,0.004997,0.005911,0.723390
3,18211,93,0.005107,0.005761,0.739248
2,18210,104,0.005711,0.005572,0.826732


Saved: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/final_evaluation/catboost_20261004_222052


## 6. Decision for the NBA stage


In [6]:
payload = json.loads(GUARD_PATH.read_text())
m = payload['test_metrics']
roc = m['raw_roc_auc']
pr_lift = m['pr_auc_lift_vs_base']
top_lift = m.get('top_decile_lift_vs_random', float('nan'))

print('Final ROC-AUC:', round(roc, 4))
print('PR-AUC lift vs base:', round(pr_lift, 3), 'x')
print('Top-decile lift:', round(top_lift, 3), 'x')

if roc >= 0.54 and top_lift > 1.0:
    print('DECISION: Freeze this model and proceed to the Next Best Action engine.')
else:
    print('DECISION: Predictive signal did not hold up strongly enough. Use segment-level historical policy for the NBA MVP rather than more tuning.')

print('\nNext notebook: 08_next_best_action_engine.ipynb')


Final ROC-AUC: 0.5745
PR-AUC lift vs base: 1.245 x
Top-decile lift: 1.335 x
DECISION: Freeze this model and proceed to the Next Best Action engine.

Next notebook: 08_next_best_action_engine.ipynb


## What this notebook produces

Under `artifacts/propensity/final_evaluation/catboost_20261004_222052/`:
- `model.joblib` — frozen selected CatBoost model
- `platt_calibrator.joblib` — probability calibrator fit on validation
- `scored_test.parquet` — one-time final test scores
- `test_deciles.csv` — holdout ranking/lift table
- `FINAL_TEST_EVALUATED.json` — final metrics + guard preventing accidental repeat evaluation

After this, do **not tune against test**. The next step is the Next Best Action engine.
